Task 1: Data Cleaning & Preprocessing

In [14]:
#Imports & Data Loading
#Imports necessary Python libraries (Pandas) and reads the raw Spotify streaming history dataset into a DataFrame using latin1 encoding to handle special characters.
import pandas as pd

# Load dataset using latin1 encoding to handle special artist/song characters
df = pd.read_csv("C:/Users/iic04/Downloads/Spotify streaming history.csv", encoding='latin1')

# View basic information
print(f"Dataset Dimensions: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()


Dataset Dimensions: 149860 rows, 11 columns


,spotify_track_uri,ts,platform,ms_played,track_name,artist_name,album_name,reason_start,reason_end,shuffle,skipped
0,2J3n32GeLmMjwuAzyhcSNe,08-07-2013 02:44,web player,3185,"Say It, Just Say It",The Mowgli's,Waiting For The Dawn,autoplay,clickrow,False,False
1,1oHxIPqJyvAYHy0PVrDU98,08-07-2013 02:45,web player,61865,Drinking from the Bottle (feat. Tinie Tempah),Calvin Harris,18 Months,clickrow,clickrow,False,False
2,487OPlneJNni3NWC8SYqhW,08-07-2013 02:50,web player,285386,Born To Die,Lana Del Rey,Born To Die - The Paradise Edition,clickrow,unknown,False,False
3,5IyblF777jLZj1vGHG2UD3,08-07-2013 02:52,web player,134022,Off To The Races,Lana Del Rey,Born To Die - The Paradise Edition,trackdone,clickrow,False,False
4,0GgAAB0ZMllFhbNc3mAodO,08-07-2013 03:17,web player,0,Half Mast,Empire Of The Sun,Walking On A Dream,clickrow,nextbtn,False,False


In [16]:
#Deduplication
#I checked the dataset for duplicate records and removed them to ensure each observation was unique.
print(df.duplicated().sum())

df.drop_duplicates(inplace=True)

print(df.duplicated().sum())

1782
0


In [20]:
#Column Management
#I removed unnecessary columns and renamed column names to improve readability and make the dataset easier to work with.

columns_to_drop = ['spotify_track_uri']
df.drop(columns=[col for col in columns_to_drop if col in df.columns], inplace=True)

#Rename columns for improved readability
df.rename(columns={
    'ts': 'timestamp',
    'ms_played': 'milliseconds_played',
    'reason_start': 'start_reason',
    'reason_end': 'end_reason',
    'shuffle': 'is_shuffle',
    'skipped': 'is_skipped'
}, inplace=True)

#Verify updated columns
print("Updated Columns:", df.columns.tolist())

Updated Columns: ['timestamp', 'platform', 'milliseconds_played', 'track_name', 'artist_name', 'album_name', 'start_reason', 'end_reason', 'is_shuffle', 'is_skipped']


In [30]:
#Missing values
#I identified missing values across the dataset and handled them using suitable imputation techniques. 
#Categorical missing values in playback reasons were imputed using the mode to keep the categorical distribution consistent.

#Checking for missing values.
df.isna().sum()

#Handling the missing values.
df["start_reason"].fillna(df["start_reason"].mode()[0], inplace=True)
df["end_reason"].fillna(df["end_reason"].mode()[0], inplace=True)

# Verify that no missing values remain
df.isna().sum()

timestamp              0
platform               0
milliseconds_played    0
track_name             0
artist_name            0
album_name             0
start_reason           0
end_reason             0
is_shuffle             0
is_skipped             0
dtype: int64

In [32]:
#Datatypes
#I converted columns into appropriate data types so that dates and numerical values could be analyzed correctly. 
#Specifically, I parsed string timestamps into standard pandas datetime objects and verified numerical playback durations.

# Convert timestamp string column into datetime format
df['timestamp'] = pd.to_datetime(df['timestamp'], format='%d-%m-%Y %H:%M')

# Ensure numeric columns are explicitly typed
df['milliseconds_played'] = pd.to_numeric(df['milliseconds_played'])

# Create numeric minutes column for easier calculation
df['minutes_played'] = df['milliseconds_played'] / (1000 * 60)

# Verify updated data types
df.dtypes

timestamp              datetime64[ns]
platform                       object
milliseconds_played             int64
track_name                     object
artist_name                    object
album_name                     object
start_reason                   object
end_reason                     object
is_shuffle                       bool
is_skipped                       bool
minutes_played                float64
dtype: object

In [34]:
#Format Standardization
#I standardized text formatting by removing extra spaces across text fields and converting platform labels to lowercase to ensure consistency throughout the dataset.

# Strip extra leading/trailing whitespace from string columns
string_cols = ['platform', 'track_name', 'artist_name', 'album_name', 'start_reason', 'end_reason']
for col in string_cols:
    df[col] = df[col].astype(str).str.strip()

# Standardize platform names to lowercase for uniform grouping
df['platform'] = df['platform'].str.lower()

# Verify platform values are clean and consistent
print("Cleaned Platform Labels:\n", df['platform'].value_counts())

Cleaned Platform Labels:
 platform
android           139002
ios                 3048
cast to device      2981
windows             1690
mac                 1176
web player           181
Name: count, dtype: int64


In [38]:
#Saving the cleaned Outcome.
#I saved the cleaned dataset as a new CSV file after verifying that duplicate records and missing values had been handled successfully.

import os

# Output filename
file_name = "cleaned_spotify_data.csv"

# Save dataframe to CSV
df.to_csv(file_name, index=False)

# Verification check to confirm the file exists
if os.path.exists(file_name):
    print(f"Dataset successfully saved as '{file_name}'! Rows: {len(df):,}")
else:
    print("Failed to save the dataset.")

Dataset successfully saved as 'cleaned_spotify_data.csv'! Rows: 148,078


Short Summary:
We systematically cleaned and preprocessed the Spotify streaming dataset following a structured data quality workflow.

First, we loaded the raw dataset using appropriate character encoding. Next, we removed duplicate rows and dropped unnecessary identifier columns like spotify_track_uri. We then checked for missing values and imputed categorical fields, specifically start_reason and end_reason, using their statistical mode. Following that, we converted timestamps into standard datetime format and explicitly parsed numeric fields like playback duration into minutes. Finally, we standardized text formatting across categorical attributes—stripping extra whitespace and converting platform labels to lowercase—and exported the clean, analysis-ready dataset to a CSV file.